# Reproducing TurboQuant's calibration-free KV-cache key quality (v1.3.0)

This notebook reproduces the core result: **matching KVQuant's 4-bit KV-cache quality on LongBench — without KVQuant's Fisher-gradient + K-means calibration.**

| KV scheme (LongBench, Llama-2-7B-chat, full 200-sample splits) | trec | triviaqa | qasper |
|---|---:|---:|---:|
| fp16 | 64.0 | 83.26 | 22.06 |
| KVQuant nuq4-1% *(Fisher + K-means)* | 64.0 | 83.16 | **21.06** |
| per-channel **uniform** 4-bit | 62.5 | 81.84 | **14.38** |
| **NF4 + 2% outliers + sink** *(no calibration)* | 63.5 | **83.32** | **20.82** |

Two parts:
1. **CPU mechanism demo (seconds)** — *why* it works, on synthetic keys, using the shipped `PerChannelKV`. No GPU, no model download.
2. **Full LongBench reproduction (GPU)** — the exact runner + commands that produced the table above.

In [ ]:
# !pip install -U turboquant-pro
import numpy as np
from turboquant_pro import PerChannelKV
print("turboquant-pro ready")

## 1. The mechanism (CPU, fast)

Attention reads `softmax(Q.K^T/sqrt(d))`, so the quality signal that matters for **keys** is the **attention-output error vs fp16** — not raw reconstruction error (which is actively misleading for keys; see `docs/KV_KEYS_FINDING.md`).

KV-cache keys are heavy-tailed: a small fraction of entries carry huge magnitudes. Uniform 4-bit spends its whole range covering them; **NF4** (a fixed NormalFloat codebook) is even *more* fragile because its per-channel abs-max scale is hijacked by a single outlier. The fix is **dense-sparse outliers**: keep the top-few-% magnitudes per channel in fp16, quantize the well-behaved bulk in 4-bit.

In [ ]:
rng = np.random.default_rng(0)
B, H, S, D = 1, 8, 512, 64

# Synthetic keys: N(0,1) with ~2% sparse heavy spikes -> the outlier structure of real post-RoPE keys
K = rng.standard_normal((B, H, S, D)).astype(np.float32)
spike = rng.random((B, H, S, D)) < 0.02
K[spike] *= rng.uniform(10, 30, size=int(spike.sum())).astype(np.float32)
V = rng.standard_normal((B, H, S, D)).astype(np.float32)
Q = rng.standard_normal((B, H, 1, D)).astype(np.float32)  # a decode-step query

def attn(Q, K, V):
    s = (Q @ K.transpose(0, 1, 3, 2)) / np.sqrt(D)
    s = s - s.max(-1, keepdims=True)
    w = np.exp(s); w /= w.sum(-1, keepdims=True)
    return w @ V

ref = attn(Q, K, V)  # fp16 reference attention output
def attn_err(Khat):  # relative error of attention output vs fp16
    return float(np.linalg.norm(attn(Q, Khat, V) - ref) / np.linalg.norm(ref))

configs = [
    ("fp16 (reference)",        None),
    ("uniform 4-bit",           PerChannelKV(D, H, bits=4)),
    ("NF4 (no outliers)",       PerChannelKV(D, H, bits=4, nf4=True)),
    ("NF4 + 1% outliers",       PerChannelKV(D, H, bits=4, nf4=True, outlier_frac=0.01)),
    ("NF4 + 2% outliers",       PerChannelKV(D, H, bits=4, nf4=True, outlier_frac=0.02)),
]
print(f"{'scheme':22s} {'attn-output err':>16s} {'recon err':>10s}")
for name, q in configs:
    if q is None:
        print(f"{name:22s} {0.0:16.4f} {0.0:10.4f}"); continue
    Khat = q.decompress(q.compress(K))
    print(f"{name:22s} {attn_err(Khat):16.4f} {np.linalg.norm(Khat-K)/np.linalg.norm(K):10.4f}")

**Read the `attn-output err` column.** NF4 *alone* is worse than uniform (its abs-max scale is hijacked by the spikes). Adding dense-sparse outliers rescues it and **beats uniform** — the same qualitative story as the real LongBench `qasper` collapse-and-recovery (14.38 -> 20.82).

## The outlier-fraction sweep — why 2% is the sweet spot

In [ ]:
print(f"{'outlier_frac':>12s} {'attn-output err':>16s} {'compression':>12s}")
for frac in [0.0, 0.01, 0.02, 0.03, 0.05]:
    q = PerChannelKV(D, H, bits=4, nf4=True, outlier_frac=frac)
    c = q.compress(K, packed=True)
    print(f"{frac:12.2f} {attn_err(q.decompress(c)):16.4f} {c.compression_ratio(D):11.2f}x")

Error drops sharply 0%->2% then flattens, while the fp16 sidecar keeps eating into the compression ratio. On real LongBench the qasper sweep is `1%->2%->3% = 20.23 -> 20.82 -> 20.67` — it **peaks at 2%**. Ship `outlier_frac=0.02`.

```python
from turboquant_pro import TurboQuantKVCache
cache = TurboQuantKVCache(key_nf4=True, key_outlier_frac=0.02)  # calibration-free
```

## 2. Full LongBench reproduction (GPU)

The headline table comes from running real `generate()` on **Llama-2-7B-chat** over full LongBench splits, with the KV cache quantized during decode. Runner: [`benchmarks/tq_enh_lb_shard.py`](tq_enh_lb_shard.py) (+ [`tq_enh_agg.py`](tq_enh_agg.py)).

**Setup** (one GPU node, ~16 GB):
```bash
pip install "transformers==4.38.2" jieba rouge fuzzywuzzy python-Levenshtein accelerate sentencepiece
# LongBench data (the HF script loader is deprecated; pull data.zip directly):
python -c "from huggingface_hub import hf_hub_download; import zipfile,os; \
p=hf_hub_download('THUDM/LongBench','data.zip',repo_type='dataset'); \
os.makedirs('/root/lb_data',exist_ok=True); zipfile.ZipFile(p).extractall('/root/lb_data')"
# clone THUDM/LongBench for config/ + metrics.py into /root/LongBench
```

**Run one variant** (env-configured; `SHARD_ID`/`NUM_SHARDS` shard samples across GPUs):
```bash
# the recommended config is NF4 + 2% outliers + sink:
CUDA_VISIBLE_DEVICES=0 SHARD_ID=0 NUM_SHARDS=1 TAG=nf4_2pct \
  KEY_BITS=4 VAL_BITS=4 GROUP=32 RESID=128 SINK=4 OUTLIER_FRAC=0.02 NUQ=1 \
  python tq_enh_lb_shard.py            # writes /root/out_nf4_2pct/<dataset>.0.jsonl
python tq_enh_agg.py nf4_2pct          # -> RESULT nf4_2pct {"trec":..., "triviaqa":..., "qasper":...}
```
Env knobs: `NUQ=1` enables NF4, `OUTLIER_FRAC` the dense-sparse fraction, `SINK` the fp16 attention-sink tokens, `NOQUANT=1` for the fp16 reference. Set `NUM_SHARDS=N` and launch N processes (one per GPU) to parallelize.

> **Honest caveats.** These are *simulation* numbers from a faithful-but-slow cache that re-quantizes the settled window each decode step (a production cache quantizes incrementally as tokens leave the hot window). The table rows are **one self-consistent harness** — comparable to each other, *not* to LongBench numbers from other setups. Full details + the 2-bit sanity row are in [`RESULTS_longbench.md`](RESULTS_longbench.md).